# HyperKT Stage 5 — Beta uncertainty head and MC dropout
**Runtime:** Google Colab T4 GPU (Python 3). **Inputs:** `hyperkt_stage2.zip`, `hyperkt_stage3b.zip`, `hyperkt_stage4.zip`. **Output:** `hyperkt_stage5.zip`.

This stage fits a Beta mean/concentration prediction head using the **training** split and selects it using **validation** AUC. It evaluates validation calibration and Monte Carlo dropout (30 passes). It does **not** access the test split. Beta concentration describes an assumed predictive distribution, not a validated Bayesian posterior; MC dropout is an approximate epistemic proxy. Stage 4 positional decay is not elapsed-time forgetting.

**Run cells sequentially.** Reuse existing Stage 4 model weights; do not claim test performance from these diagnostics.

In [ ]:
import os, io, json, zipfile, copy, random, shutil, math
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from google.colab import files
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'PyTorch:',torch.__version__)
assert DEVICE.type=='cuda','Choose Runtime > Change runtime type > T4 GPU'


In [ ]:
uploaded=files.upload()
required=['hyperkt_stage2.zip','hyperkt_stage3b.zip','hyperkt_stage4.zip']
assert all(x in uploaded for x in required),f'Missing {[x for x in required if x not in uploaded]}'
z2,z3,z4=[zipfile.ZipFile(io.BytesIO(uploaded[x])) for x in required]
def load_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as f:return {k:f[k].copy() for k in f.files}
train=load_npz(z2,'train_sequences.npz');val=load_npz(z2,'val_sequences.npz')
hg=load_npz(z3,'hypergraph_repaired.npz')
assert hg['assignments'].shape==(149,32)
assert set(train['user_id']).isdisjoint(set(val['user_id']))
print('Train targets:',int(train['mask'].sum()),'Val targets:',int(val['mask'].sum()))


In [ ]:
PAD=1;N_SKILLS=151;MAXLEN=200
A=np.zeros((N_SKILLS,32),dtype=np.float32)
A[hg['skill_ids'].astype(int)]=hg['assignments'].astype(np.float32)
A[PAD]=0;A=torch.tensor(A)
def dataset(d):
    names=['history_skill','history_correct','target_skill','target_correct','mask']
    return TensorDataset(*(torch.as_tensor(d[k],dtype=torch.bool if k=='mask' else torch.long) for k in names))
train_loader=DataLoader(dataset(train),batch_size=16,shuffle=True,num_workers=0)
val_loader=DataLoader(dataset(val),batch_size=16,shuffle=False,num_workers=0)
class HyperKTTemporal(nn.Module):
    def __init__(self,use_hypergraph=False,d=256,heads=8,layers=4,dropout=.1):
        super().__init__();self.use_hypergraph=use_hypergraph
        self.skill=nn.Embedding(N_SKILLS,d,padding_idx=PAD)
        self.response=nn.Embedding(2,d);self.pos=nn.Embedding(MAXLEN,d)
        self.decay_raw=nn.Parameter(torch.tensor(-2.97))
        if use_hypergraph:
            self.register_buffer('assignments',A.clone())
            self.hg_proj=nn.Linear(32,d,bias=False);self.hg_gate=nn.Linear(2*d,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=dropout,batch_first=True,activation='gelu',norm_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d)
        self.head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(dropout),nn.Linear(d,1))
    def features(self,hs,hc,ts,mask):
        B,L=hs.shape;pos=torch.arange(L,device=hs.device)
        age=(L-1-pos).float();decay=torch.exp(-F.softplus(self.decay_raw)*age/MAXLEN)
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)[None,:,:]*decay[None,:,None]
        if self.use_hypergraph:
            structure=self.hg_proj(self.assignments[hs]);gate=torch.sigmoid(self.hg_gate(torch.cat([x,structure],dim=-1)))
            x=x+gate*structure
        causal=torch.triu(torch.ones(L,L,device=hs.device,dtype=torch.bool),diagonal=1)
        x=x.masked_fill(~mask.unsqueeze(-1),0.0)
        h=self.norm(self.encoder(x,mask=causal))
        target=self.skill(ts)
        if self.use_hypergraph:target=target+self.hg_proj(self.assignments[ts])
        return torch.cat([h,target],dim=-1)
    def forward(self,hs,hc,ts,mask):return self.head(self.features(hs,hc,ts,mask)).squeeze(-1)


In [ ]:
# Load Stage 4 checkpoint without importing arbitrary pickle objects; only trusted project-generated ZIPs.
ckpt=torch.load(io.BytesIO(z4.read('hypergraph_fusion.pt')),map_location='cpu',weights_only=True)
assert ckpt['model']=='hypergraph_fusion'
base=HyperKTTemporal(use_hypergraph=True).to(DEVICE)
base.load_state_dict(ckpt['state_dict'],strict=True)
print('Loaded Stage 4 fusion checkpoint, best epoch:',ckpt['best_epoch'])
class BetaHyperKT(nn.Module):
    def __init__(self,backbone):
        super().__init__();self.backbone=backbone
        # Initialize mean logit from trained Stage 4 output head; concentration is learned separately.
        self.mean_head=copy.deepcopy(backbone.head)
        self.concentration_head=nn.Sequential(nn.Linear(512,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
    def forward(self,hs,hc,ts,mask):
        f=self.backbone.features(hs,hc,ts,mask)
        mu=torch.sigmoid(self.mean_head(f).squeeze(-1))
        kappa=2.0+F.softplus(self.concentration_head(f).squeeze(-1))
        return mu,kappa
model=BetaHyperKT(base).to(DEVICE)
# Freeze backbone for an interpretable Stage 5 head-only experiment; unfreeze in Stage 6.
for p in model.backbone.parameters():p.requires_grad_(False)
# Training the new head only (mean head initialized from Stage 4)
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=2e-4,weight_decay=.01)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# Beta-Bernoulli predictive mean = mu. Bernoulli NLL alone cannot identify kappa.
# A proper *per-observation* Beta-Bernoulli likelihood is independent of concentration.
# To avoid falsely claiming learned concentration, fit the mean head using BCE and
# leave concentration untrained; report concentration as illustrative ONLY.
# MC dropout variation supplies a separate uncertainty diagnostic.
def epoch_train():
    model.train();tot=cnt=0
    for hs,hc,ts,y,m in train_loader:
        hs,hc,ts,y,m=[v.to(DEVICE) for v in (hs,hc,ts,y,m)]
        optimizer.zero_grad(set_to_none=True)
        mu,kappa=model(hs,hc,ts,m)
        loss=F.binary_cross_entropy(mu[m].clamp(1e-6,1-1e-6),y[m].float())
        loss.backward();nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        optimizer.step();tot+=float(loss.detach())*int(m.sum());cnt+=int(m.sum())
    return tot/cnt
@torch.no_grad()
def predict(loader,mc=1):
    # For MC passes, enable dropout while keeping model weights fixed.
    model.train(mc>1)
    yy=[]; pp=[]; uu=[]
    for hs,hc,ts,y,m in loader:
        hs,hc,ts,y,m=[v.to(DEVICE) for v in (hs,hc,ts,y,m)]
        probs=[]
        for _ in range(mc):
            mu,_=model(hs,hc,ts,m)
            probs.append(mu[m].detach().cpu().numpy())
        probs=np.stack(probs,axis=0)
        yy.append(y[m].cpu().numpy());pp.append(probs.mean(0));uu.append(probs.var(0,ddof=0))
    return np.concatenate(yy),np.concatenate(pp),np.concatenate(uu)
def ece(y,p,n_bins=15):
    bins=np.linspace(0,1,n_bins+1); ids=np.minimum(np.searchsorted(bins,p,side='right')-1,n_bins-1)
    total=len(y);return float(sum((ids==i).sum()/total*abs(y[ids==i].mean()-p[ids==i].mean()) for i in range(n_bins) if (ids==i).any()))
def metrics(y,p):
    p=np.clip(p,1e-7,1-1e-7)
    return {'auc':float(roc_auc_score(y,p)),'nll':float(log_loss(y,p,labels=[0,1])),
            'brier':float(brier_score_loss(y,p)),'ece_15':ece(y,p),'n_targets':int(len(y))}


In [ ]:
# Validation-only model selection. No test data read.
EPOCHS=8;PATIENCE=3;best_auc=-1;best_epoch=None;best_state=None;history=[];wait=0
for epoch in range(1,EPOCHS+1):
    tr=epoch_train();y,p,_=predict(val_loader,mc=1);v=metrics(y,p)
    history.append({'epoch':epoch,'train_nll':tr,**{'val_'+k:x for k,x in v.items()}})
    print('epoch',epoch,'train NLL',round(tr,5),'val AUC',round(v['auc'],5),'val ECE',round(v['ece_15'],5),flush=True)
    if v['auc']>best_auc+1e-5:
        best_auc=v['auc'];best_epoch=epoch;wait=0
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        wait+=1
        if wait>=PATIENCE:break
model.load_state_dict(best_state)
print('Selected epoch',best_epoch)


In [ ]:
# MC dropout, 30 passes on validation only.
y,p_det,_=predict(val_loader,mc=1)
y2,p_mc,var_mc=predict(val_loader,mc=30)
assert np.array_equal(y,y2)
report={'stage':'5','seed':SEED,'selected_epoch':best_epoch,'deterministic_validation':metrics(y,p_det),
        'mc30_validation':metrics(y2,p_mc),'mc30_variance_mean':float(var_mc.mean()),
        'mc30_variance_p95':float(np.percentile(var_mc,95)),
        'interpretation':'MC dropout variance is an approximate uncertainty diagnostic. Beta concentration is not identifiable from binary BCE and must not be reported as calibrated uncertainty.',
        'test_used':False,'stage4_reference_val_auc':float(json.loads(z4.read('report.json'))['hypergraph_fusion']['best_val_auc'])}
print(json.dumps(report,indent=2))


In [ ]:
# Reliability plot and portable research artifacts.
import matplotlib.pyplot as plt
out='/content/hyperkt_stage5';os.makedirs(out,exist_ok=True)
plt.figure(figsize=(5,5));plt.plot([0,1],[0,1],'--',label='Perfect calibration')
for name,p in [('Deterministic',p_det),('MC30',p_mc)]:
    bins=np.linspace(0,1,16);ids=np.minimum(np.searchsorted(bins,p,side='right')-1,14)
    xx=[p[ids==i].mean() for i in range(15) if (ids==i).any()]
    yy=[y[ids==i].mean() for i in range(15) if (ids==i).any()]
    plt.plot(xx,yy,marker='o',label=name)
plt.xlabel('Mean predicted probability');plt.ylabel('Observed correctness');plt.legend();plt.tight_layout()
plt.savefig(f'{out}/validation_reliability.png',dpi=180);plt.close()
pd.DataFrame(history).to_csv(f'{out}/training_history.csv',index=False)
np.savez_compressed(f'{out}/validation_predictions.npz',y=y,p_deterministic=p_det,p_mc30=p_mc,mc30_variance=var_mc)
torch.save({'state_dict':best_state,'seed':SEED,'best_epoch':best_epoch,'backbone':'hypergraph_fusion',
            'training':'frozen_backbone_mean_head_BCE','beta_concentration_identifiable':False},f'{out}/stage5_head.pt')
with open(f'{out}/report.json','w') as f:json.dump(report,f,indent=2)
with open(f'{out}/protocol.txt','w') as f:f.write('Student-disjoint validation only; frozen Stage 4 fusion backbone; mean-head BCE; 30-pass MC dropout; no test access. Beta concentration not identifiable from binary BCE.\n')
shutil.make_archive('/content/hyperkt_stage5','zip',out)
print('Saved /content/hyperkt_stage5.zip');files.download('/content/hyperkt_stage5.zip')


## Interpretation and next stage
Compare Stage 5 validation AUC and calibration to Stage 4. Do not report MC-dropout variance as validated epistemic uncertainty, nor Beta concentration as learned from Bernoulli labels. Stage 6 should decide whether to use a principled hierarchical Beta model or retain the simpler Bernoulli + MC dropout approach, then train full HyperKT and run repeated seeds before opening the held-out test set.